# Maximum Likelihood Estimation from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/maximum_likelihood_estimation.ipynb)

**Companion blog post:** [Maximum Likelihood Estimation from Scratch](https://sesen.ai/blog/maximum-likelihood-estimation-from-scratch)

---

In this notebook, you'll implement Maximum Likelihood Estimation (MLE) from scratch across three distributions:

1. **Bernoulli/Binomial** — Estimating a coin's bias
2. **Normal (Gaussian)** — Estimating mean and standard deviation
3. **Multinomial** — Estimating category probabilities

By the end, you'll understand:
- What likelihood and log-likelihood are
- Why we use log-likelihood (numerical stability)
- How to find the MLE analytically and numerically
- How MLE connects to the EM algorithm

## 1. Bernoulli MLE: The Coin Flip

The simplest MLE problem: you flip a coin `n` times and observe `k` heads. What's the coin's bias θ?

The likelihood function is:

$$\mathcal{L}(\theta) = \theta^k (1-\theta)^{n-k}$$

Let's compute and visualise it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log, factorial, pi

# Observed data: 73 heads out of 100 flips
n_heads = 73
n_tails = 27
n_total = n_heads + n_tails

# Compute likelihood for every possible bias value
theta_values = np.linspace(0.01, 0.99, 200)
likelihoods = theta_values**n_heads * (1 - theta_values)**n_tails

# The MLE is simply the proportion of heads
theta_mle = n_heads / n_total

plt.figure(figsize=(8, 4))
plt.plot(theta_values, likelihoods / likelihoods.max(), 'b-', linewidth=2)
plt.axvline(x=theta_mle, color='r', linestyle='--', label=f'MLE: θ = {theta_mle:.2f}')
plt.xlabel('θ (coin bias)')
plt.ylabel('Likelihood (normalised)')
plt.title('Likelihood Function for a Coin Flip')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"MLE estimate: θ = {theta_mle:.2f}")

### Why Log-Likelihood?

Watch what happens when we compute the raw likelihood:

In [ ]:
# Raw likelihood is astronomically small
raw_likelihood = 0.73**73 * 0.27**27
print(f"Raw likelihood: {raw_likelihood:.2e}")
print(f"Log-likelihood: {73 * np.log(0.73) + 27 * np.log(0.27):.4f}")
print()
print("With 1000 data points, raw likelihood would underflow to 0.0")
print(f"Example: 0.73^730 * 0.27^270 = {0.73**730 * 0.27**270}")
print(f"But log-likelihood is fine: {730 * np.log(0.73) + 270 * np.log(0.27):.4f}")

### Deriving the MLE Analytically

The log-likelihood for the Bernoulli is:

$$\ell(\theta) = k \log \theta + (n-k) \log(1-\theta)$$

Taking the derivative and setting to zero:

$$\frac{d\ell}{d\theta} = \frac{k}{\theta} - \frac{n-k}{1-\theta} = 0$$

$$\hat{\theta}_{\text{MLE}} = \frac{k}{n}$$

Let's verify this computationally:

In [ ]:
# Compute log-likelihood across theta values
log_likelihoods = n_heads * np.log(theta_values) + n_tails * np.log(1 - theta_values)

# Find the theta that maximises log-likelihood
theta_mle_numerical = theta_values[np.argmax(log_likelihoods)]
theta_mle_analytical = n_heads / n_total

plt.figure(figsize=(8, 4))
plt.plot(theta_values, log_likelihoods, 'b-', linewidth=2)
plt.axvline(x=theta_mle_analytical, color='r', linestyle='--',
            label=f'Analytical MLE: {theta_mle_analytical:.2f}')
plt.xlabel('θ')
plt.ylabel('Log-Likelihood')
plt.title('Log-Likelihood for Bernoulli')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Analytical MLE: {theta_mle_analytical:.4f}")
print(f"Numerical MLE:  {theta_mle_numerical:.4f}")

## 2. Normal Distribution MLE

Now let's estimate two parameters: mean μ and standard deviation σ.

The log-likelihood for n observations from N(μ, σ²):

$$\ell(\mu, \sigma) = -\frac{n}{2}\log(2\pi) - n\log\sigma - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(x_i - \mu)^2$$

In [ ]:
def normal_log_likelihood(data, mu, sigma):
    """Compute log-likelihood of data under a Normal distribution."""
    n = len(data)
    ll = -0.5 * n * log(2 * pi) - n * log(sigma)
    ll -= 0.5 * sum((x - mu)**2 / sigma**2 for x in data)
    return ll


def normal_log_likelihood_fast(data, mu, sigma):
    """Vectorised log-likelihood (ignoring constant offset)."""
    n = len(data)
    residuals = data - mu
    return -0.5 * (n * np.log(sigma**2) + np.sum(residuals**2 / sigma**2))


# Generate data from N(1, 1)
np.random.seed(42)
data = np.random.normal(loc=1.0, scale=1.0, size=10_000)

# Compare different parameter guesses
guesses = [(0.5, 2.0), (1.0, 3.0), (1.0, 1.0)]

print("Log-likelihood for different parameter guesses:")
print("=" * 50)
for mu_est, sigma_est in guesses:
    ll = normal_log_likelihood_fast(data, mu_est, sigma_est)
    print(f"  μ={mu_est:.1f}, σ={sigma_est:.1f}  →  ℓ = {ll:.1f}")

print(f"\nTrue values (μ=1, σ=1) give the highest log-likelihood!")

### Numerical Optimisation with scipy

When analytical solutions don't exist, we can use numerical optimisation.
We minimise the *negative* log-likelihood (since optimisers minimise by default):

In [ ]:
from scipy.optimize import minimize

# Start from a bad guess
x0 = np.array([0.5, 2.0])  # [mu_guess, sigma_guess]

result = minimize(
    lambda params: -normal_log_likelihood_fast(data, params[0], params[1]),
    x0,
    method='nelder-mead',
    options={'xatol': 1e-8}
)

print(f"True:      μ = 1.000, σ = 1.000")
print(f"Numerical: μ = {result.x[0]:.3f}, σ = {result.x[1]:.3f}")
print(f"Analytic:  μ = {data.mean():.3f}, σ = {data.std():.3f}")
print(f"\nOptimiser converged in {result.nit} iterations")

### Visualising the Likelihood Surface

With two parameters, the likelihood becomes a surface. The contour plot shows it's concave with a single peak:

In [ ]:
mu_range = np.linspace(0.5, 1.5, 100)
sigma_range = np.linspace(0.7, 1.3, 100)
MU, SIGMA = np.meshgrid(mu_range, sigma_range)

LL = np.zeros_like(MU)
for i in range(len(mu_range)):
    for j in range(len(sigma_range)):
        LL[j, i] = normal_log_likelihood_fast(data, MU[j, i], SIGMA[j, i])

plt.figure(figsize=(8, 6))
plt.contourf(MU, SIGMA, LL, levels=30, cmap='viridis')
plt.colorbar(label='Log-Likelihood')
plt.plot(data.mean(), data.std(), 'r*', markersize=15, label='MLE')
plt.xlabel('μ')
plt.ylabel('σ')
plt.title('Log-Likelihood Surface for Normal Distribution')
plt.legend()
plt.show()

## 3. Multinomial MLE

Now let's tackle a distribution with multiple parameters. A multinomial models k possible outcomes, each with probability p₁, p₂, ..., pₖ.

The log-likelihood for a single observation:

$$\log P(\mathbf{x} \mid \mathbf{p}) = \log\binom{n}{x_1, \ldots, x_k} + \sum_{i=1}^{k} x_i \log p_i$$

In [ ]:
def multinomial_log_likelihood(obs, probs):
    """Compute log-likelihood for a single multinomial observation."""
    n = sum(obs)
    # Multinomial coefficient: n! / (x1! * x2! * ... * xk!)
    log_coeff = log(factorial(n)) - sum(log(factorial(x)) for x in obs)
    # Probability term: sum(xi * log(pi)), skip zero counts to avoid log(0)
    log_prob = sum(x * log(p) for x, p in zip(obs, probs) if x > 0)
    return log_coeff + log_prob


def total_log_likelihood(data, probs):
    """Sum log-likelihood across all observations."""
    return sum(multinomial_log_likelihood(obs, probs) for obs in data)


# Generate data from a 3-state multinomial: P = [0.5, 0.2, 0.3]
np.random.seed(42)
true_probs = [0.5, 0.2, 0.3]
mn_data = np.random.multinomial(1, true_probs, size=100)

# Show a few observations
print("First 5 observations (one-hot encoded):")
for i in range(5):
    print(f"  Obs {i+1}: {mn_data[i]}  (category {np.argmax(mn_data[i]) + 1})")

sample_probs = mn_data.sum(axis=0) / mn_data.sum()
print(f"\nObserved frequencies: {mn_data.sum(axis=0)}")
print(f"Sample proportions: [{sample_probs[0]:.3f}, {sample_probs[1]:.3f}, {sample_probs[2]:.3f}]")

In [ ]:
# Grid search over (p1, p2), with p3 = 1 - p1 - p2
best_ll = -np.inf
best_probs = None

for p1 in np.arange(0.05, 0.95, 0.05):
    for p2 in np.arange(0.05, 0.95 - p1, 0.05):
        p3 = 1 - p1 - p2
        if p3 > 0:
            ll = total_log_likelihood(mn_data, [p1, p2, p3])
            if ll > best_ll:
                best_ll = ll
                best_probs = [p1, p2, p3]

# Compare with the analytical MLE (sample proportions)
sample_probs = mn_data.sum(axis=0) / mn_data.sum()

print(f"True:     P = [{true_probs[0]:.2f}, {true_probs[1]:.2f}, {true_probs[2]:.2f}]")
print(f"Grid MLE: P = [{best_probs[0]:.2f}, {best_probs[1]:.2f}, {best_probs[2]:.2f}]")
print(f"Analytic: P = [{sample_probs[0]:.2f}, {sample_probs[1]:.2f}, {sample_probs[2]:.2f}]")

### Visualising the Multinomial Likelihood Surface

With 3 categories, we have 2 free parameters. Let's plot the log-likelihood as a function of (p₁, p₂):

In [ ]:
# Create grid over (p1, p2) with p3 = 1 - p1 - p2
p1_range = np.linspace(0.05, 0.90, 80)
p2_range = np.linspace(0.05, 0.90, 80)
P1, P2 = np.meshgrid(p1_range, p2_range)

LL_mn = np.full_like(P1, np.nan)
for i in range(len(p1_range)):
    for j in range(len(p2_range)):
        p3 = 1 - P1[j, i] - P2[j, i]
        if p3 > 0.01:  # Valid probability simplex
            LL_mn[j, i] = total_log_likelihood(
                mn_data, [P1[j, i], P2[j, i], p3]
            )

plt.figure(figsize=(8, 6))
plt.contourf(P1, P2, LL_mn, levels=30, cmap='viridis')
plt.colorbar(label='Log-Likelihood')
plt.plot(sample_probs[0], sample_probs[1], 'r*', markersize=15, label='MLE')
plt.plot(true_probs[0], true_probs[1], 'w+', markersize=15, markeredgewidth=2, label='True')
plt.xlabel('p₁')
plt.ylabel('p₂')
plt.title('Log-Likelihood Surface for Multinomial (p₃ = 1 - p₁ - p₂)')
plt.legend()
plt.show()

## 4. The MLE Pattern

Notice the pattern across all three distributions:

| Distribution | Parameters | MLE |
|-------------|-----------|-----|
| Bernoulli | θ (bias) | Proportion of successes |
| Normal | μ, σ | Sample mean, sample std |
| Multinomial | p₁, ..., pₖ | Sample proportions |

MLE often gives you the "obvious" answer. But the framework proves *why* these are optimal.

## 5. Effect of Sample Size

How does the number of observations affect our estimates? The MLE converges at rate 1/√n:

In [ ]:
# Effect of sample size on MLE accuracy
true_theta = 0.73
sample_sizes = [10, 30, 100, 300, 1000, 3000, 10000]
n_trials = 200

mle_means = []
mle_stds = []

np.random.seed(42)
for n in sample_sizes:
    estimates = []
    for _ in range(n_trials):
        flips = np.random.binomial(1, true_theta, size=n)
        theta_hat = flips.mean()
        estimates.append(theta_hat)
    mle_means.append(np.mean(estimates))
    mle_stds.append(np.std(estimates))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bias (should be ~0)
axes[0].semilogx(sample_sizes, [m - true_theta for m in mle_means], 'bo-')
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Sample Size (n)')
axes[0].set_ylabel('Bias (MLE - true)')
axes[0].set_title('MLE is Unbiased')
axes[0].grid(True, alpha=0.3)

# Standard deviation (should follow 1/sqrt(n))
theoretical_std = [np.sqrt(true_theta * (1 - true_theta) / n) for n in sample_sizes]
axes[1].loglog(sample_sizes, mle_stds, 'bo-', label='Empirical std')
axes[1].loglog(sample_sizes, theoretical_std, 'r--', label='Theoretical 1/√n')
axes[1].set_xlabel('Sample Size (n)')
axes[1].set_ylabel('Std of MLE')
axes[1].set_title('MLE Variance Decreases as 1/n')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Exercises

Try these to deepen your understanding:

In [ ]:
# Exercise 1: Poisson MLE
# The Poisson distribution models count data: P(X=k) = λ^k * e^(-λ) / k!
# Derive the MLE for λ, then verify it numerically.
#
# Hint: The log-likelihood is:
#   ℓ(λ) = sum(xi) * log(λ) - n*λ - sum(log(xi!))
#
# What is dℓ/dλ = 0?

# Your code here:
# np.random.seed(42)
# poisson_data = np.random.poisson(lam=3.5, size=1000)
# lambda_mle = ...

In [ ]:
# Exercise 2: MLE vs MAP
# Add a Beta prior to the coin flip problem.
# The MAP estimate with Beta(a, b) prior is:
#   θ_MAP = (k + a - 1) / (n + a + b - 2)
#
# Compare MLE and MAP for:
# - n=3, k=3 (3 heads out of 3 flips)
# - Use Beta(2, 2) as a mild prior toward 0.5

# Your code here:
# n, k = 3, 3
# theta_mle = ...
# theta_map = ...

In [ ]:
# Exercise 3: Confidence Intervals via Fisher Information
# For the Bernoulli, the Fisher Information is:
#   I(θ) = n / (θ(1-θ))
#
# The approximate 95% confidence interval is:
#   θ_MLE ± 1.96 / sqrt(I(θ_MLE))
#
# Compute this for our coin flip example (n=100, k=73)

# Your code here:
# fisher_info = ...
# ci_width = ...

In [ ]:
# Exercise 4: Exponential Distribution MLE
# The Exponential distribution models waiting times: f(x|λ) = λ * e^(-λx)
# Derive the MLE for λ and implement it.
#
# Generate data: np.random.exponential(scale=1/λ, size=n)
# Note: numpy uses scale = 1/λ, not λ directly

# Your code here:

In [ ]:
# Exercise 5: When MLE Breaks
# Try fitting a Normal distribution to data with an outlier.
# Generate 99 points from N(0, 1) and add one point at x=100.
# How does the outlier affect the MLE for μ and σ?
#
# Compare with the median (a robust alternative to the mean).

# Your code here:
# clean_data = np.random.normal(0, 1, 99)
# dirty_data = np.append(clean_data, 100)
# ...

## 7. Summary

### Key Takeaways

1. **Likelihood ≠ Probability** — Same formula, different perspective (fixed params vs fixed data)
2. **Always use log-likelihood** — Products become sums, numerical stability guaranteed
3. **MLE = argmax of likelihood** — Find it analytically (set derivative to 0) or numerically (scipy.optimize)
4. **MLE is optimal** — Consistent, efficient, and sufficient (under regularity conditions)
5. **MLE has limits** — Overfits with small data, only gives point estimates, needs complete data

### What's Next?

- **EM Algorithm** — When data has hidden variables, MLE can't be computed directly. The [EM algorithm](https://sesen.ai/blog/em-algorithm-coin-toss-intuitive-guide) extends MLE to handle this.
- **MCMC** — When you want full posterior distributions instead of point estimates, [Metropolis-Hastings](https://sesen.ai/blog/mcmc-metropolis-hastings-island-hopping-guide) provides a Bayesian alternative.

---

**Author:** Dr. Berkan Sesen | [sesen.ai](https://sesen.ai)

**Companion blog post:** [Maximum Likelihood Estimation from Scratch](https://sesen.ai/blog/maximum-likelihood-estimation-from-scratch)